In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All project files will live here (persists between sessions)
PROJECT_DIR = '/content/drive/MyDrive/ir_project'

os.makedirs(f'{PROJECT_DIR}/corpus',  exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models',  exist_ok=True)

# Change working directory to project
os.chdir(PROJECT_DIR)
print("✅ Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Working directory: /content/drive/MyDrive/ir_project


In [11]:
!pip install streamlit rank_bm25 sentence-transformers pyspellchecker nltk pyngrok scikit-learn pandas numpy torch --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ All packages installed")

✅ All packages installed


In [12]:
from google.colab import files
import shutil
import pandas as pd

print("📂 Upload your corpus.csv file now:")
uploaded = files.upload()  # A file picker will appear

# Move it to the corpus folder
for filename in uploaded.keys():
    shutil.move(filename, f'{PROJECT_DIR}/corpus/corpus.csv')
    print(f"✅ Saved as: {PROJECT_DIR}/corpus/corpus.csv")

# Load and preview it to check field names
df = pd.read_csv(f'{PROJECT_DIR}/corpus/corpus.csv')

print(f"\n📊 Total documents (rows): {len(df)}")
print(f"🔑 Column names in your corpus: {list(df.columns)}")
print(f"\n📄 First document preview:")
print(df.iloc[0])

📂 Upload your corpus.csv file now:


Saving balanced_corpus_final (1).csv to balanced_corpus_final (1).csv
✅ Saved as: /content/drive/MyDrive/ir_project/corpus/corpus.csv

📊 Total documents (rows): 506
🔑 Column names in your corpus: ['title', 'text', 'link', 'topic', 'type', 'doc_length']

📄 First document preview:
title                                   Artificial intelligence
text          Artificial intelligence (AI) is the capability...
link          https://en.wikipedia.org/wiki/Artificial_intel...
topic                                   Artificial Intelligence
type                                                  Wikipedia
doc_length                                                12886
Name: 0, dtype: object


In [13]:
%%writefile corpus_loader.py

import json, os, pandas as pd

def load_corpus(corpus_path):
    ext = os.path.splitext(corpus_path)[1].lower()

    if ext == '.json':
        with open(corpus_path, 'r', encoding='utf-8') as f:
            documents = json.load(f)

    elif ext == '.csv':
        documents = pd.read_csv(corpus_path).to_dict('records')

    else:
        raise ValueError(f"Unsupported format: {ext}")

    normalized = []
    for i, doc in enumerate(documents):
        normalized.append({
            # Unique ID (generate if missing)
            'id': doc.get('id', f'doc_{i:04d}'),

            # Core fields
            'title': doc.get('title', 'Untitled'),
            'text': doc.get('text', ''),
            'topic': doc.get('topic', 'Unknown'),
            'link': doc.get('link', f'https://corpus.local/doc/{i}'),

            # Additional fields from your CSV
            'type': doc.get('type', ''),
            'doc_length': doc.get('doc_length', 0),
        })

    print(f"Loaded {len(normalized)} documents.")
    return normalized

Overwriting corpus_loader.py


In [14]:
%%writefile query_handler.py

import re
from spellchecker import SpellChecker

ABBREVIATIONS = {
    'ai': 'artificial intelligence', 'ml': 'machine learning',
    'dl': 'deep learning', 'nlp': 'natural language processing',
    'iot': 'internet of things', 'db': 'database',
    'os': 'operating system', 'api': 'application programming interface',
    'sql': 'structured query language', 'nn': 'neural network',
    'cv': 'computer vision', 'rl': 'reinforcement learning',
    'llm': 'large language model', 'gpu': 'graphics processing unit',
    'cpu': 'central processing unit', 'p2p': 'peer to peer',
    'vpn': 'virtual private network', 'ddos': 'distributed denial of service',
    'nft': 'non fungible token', 'defi': 'decentralized finance',
    'qc': 'quantum computing',
}

class QueryHandler:
    def __init__(self):
        self.spell = SpellChecker()

    def expand_abbreviations(self, query):
        words = query.split()
        expanded = []
        for word in words:
            clean = word.lower().strip("'\".,!?")
            expanded.append(ABBREVIATIONS[clean] if clean in ABBREVIATIONS else word)
        return ' '.join(expanded)

    def correct_spelling(self, query):
        phrase_pattern = re.compile(r'"[^"]+"')
        phrases = phrase_pattern.findall(query)
        clean_query = query
        for i, phrase in enumerate(phrases):
            clean_query = clean_query.replace(phrase, f'__PHRASE_{i}__')

        words = clean_query.split()
        corrected_words, corrections = [], {}
        for word in words:
            if word.startswith('__PHRASE_') or word.lower() in ABBREVIATIONS or len(word) <= 2:
                corrected_words.append(word)
                continue
            correction = self.spell.correction(word)
            if correction and correction != word.lower():
                corrections[word] = correction
                corrected_words.append(correction)
            else:
                corrected_words.append(word)

        corrected_query = ' '.join(corrected_words)
        for i, phrase in enumerate(phrases):
            corrected_query = corrected_query.replace(f'__PHRASE_{i}__', phrase)
        return corrected_query, corrections

    def extract_phrase(self, query):
        match = re.search(r'"([^"]+)"', query)
        return match.group(1) if match else None

    def process_query(self, query):
        query_lower = query.lower()
        corrected, corrections = self.correct_spelling(query_lower)
        expanded = self.expand_abbreviations(corrected)
        phrase = self.extract_phrase(expanded)
        return {
            'original': query, 'processed': expanded,
            'corrections': corrections, 'phrase': phrase, 'expanded': expanded,
        }

Overwriting query_handler.py


In [15]:
%%writefile models/__init__.py
# empty — marks this folder as a Python package

Overwriting models/__init__.py


In [16]:
%%writefile models/vsm.py

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class VSMRetriever:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
        self.tfidf_matrix = None
        self.documents = None

    def fit(self, documents):
        self.documents = documents
        corpus = [d['title'] + ' ' + d['text'] for d in documents]
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
        print(f"VSM: Indexed {len(documents)} documents.")

    def search(self, query, top_k=10, phrase=None):
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        if phrase:
            for i, doc in enumerate(self.documents):
                if phrase.lower() in (doc['title']+' '+doc['content']).lower():
                    scores[i] *= 1.5
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{'doc': self.documents[i], 'score': float(scores[i]), 'rank': r+1}
                for r, i in enumerate(top_indices) if scores[i] > 0]

Overwriting models/vsm.py


In [17]:
%%writefile models/bm25.py

import numpy as np, re, nltk
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

class BM25Retriever:
    def __init__(self, k1=1.5, b=0.75):
        self.k1=k1; self.b=b; self.bm25=None; self.documents=None

    def _tokenize(self, text):
        text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
        return word_tokenize(text)

    def fit(self, documents):
        self.documents = documents
        tokenized = [
        self._tokenize(d['title'] + ' ' + d['text']) for d in documents]
        self.bm25 = BM25Okapi(tokenized, k1=self.k1, b=self.b)
        print(f"BM25: Indexed {len(documents)} documents.")

    def search(self, query, top_k=10, phrase=None):
        scores = self.bm25.get_scores(self._tokenize(query))
        if phrase:
            for i, doc in enumerate(self.documents):
                if phrase.lower() in (doc['title'] + ' ' + doc['text']).lower():
                      scores[i] *= 1.5
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{'doc': self.documents[i], 'score': float(scores[i]), 'rank': r+1}
                for r, i in enumerate(top_indices) if scores[i] > 0]

Overwriting models/bm25.py


In [18]:
%%writefile models/embedding.py

import numpy as np, pickle, os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

class EmbeddingRetriever:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name=model_name; self.model=None
        self.embeddings=None; self.documents=None

    def fit(self, documents, cache_path='embeddings_cache.pkl'):
        self.documents = documents
        if os.path.exists(cache_path):
            print("Embeddings: Loading from cache...")
            with open(cache_path,'rb') as f:
                self.embeddings = pickle.load(f)
        else:
            print("Embeddings: Computing (first time, ~2-4 min)...")
            self.model = SentenceTransformer(self.model_name)
            texts = [d['title'] + ' ' + d['text'][:512] for d in documents]
            self.embeddings = self.model.encode(texts, show_progress_bar=True, batch_size=32)
            with open(cache_path,'wb') as f:
                pickle.dump(self.embeddings, f)
            print("Saved to cache.")
        if self.model is None:
            self.model = SentenceTransformer(self.model_name)
        print(f"Embeddings: Ready. ({len(documents)} docs)")

    def search(self, query, top_k=10, phrase=None):
        query_emb = self.model.encode([query])
        scores = cosine_similarity(query_emb, self.embeddings).flatten()
        if phrase:
            for i, doc in enumerate(self.documents):
                if phrase.lower() in (doc['title'] + ' ' + doc['text']).lower():
                    scores[i] *= 1.3
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{'doc': self.documents[i], 'score': float(scores[i]), 'rank': r+1}
                for r, i in enumerate(top_indices)]

Overwriting models/embedding.py


In [19]:
%%writefile evaluator.py

import numpy as np

class Evaluator:
    def precision_at_k(self, retrieved_ids, relevant_ids, k=10):
        hits = sum(1 for d in retrieved_ids[:k] if d in set(relevant_ids))
        return hits / k

    def recall(self, retrieved_ids, relevant_ids):
        if not relevant_ids: return 0.0
        return sum(1 for d in retrieved_ids if d in set(relevant_ids)) / len(relevant_ids)

    def average_precision(self, retrieved_ids, relevant_ids):
        relevant_set = set(relevant_ids)
        hits = 0; sum_p = 0.0
        for i, doc_id in enumerate(retrieved_ids):
            if doc_id in relevant_set:
                hits += 1; sum_p += hits / (i + 1)
        return sum_p / len(relevant_ids) if relevant_ids else 0.0

    def evaluate_model(self, model_name, queries_results):
        p10s, recalls, aps = [], [], []
        for qr in queries_results:
            p10s.append(self.precision_at_k(qr['retrieved_ids'], qr['relevant_ids']))
            recalls.append(self.recall(qr['retrieved_ids'], qr['relevant_ids']))
            aps.append(self.average_precision(qr['retrieved_ids'], qr['relevant_ids']))
        return {
            'model': model_name,
            'avg_precision@10': np.mean(p10s),
            'avg_recall': np.mean(recalls),
            'MAP': np.mean(aps),
            'per_query_p10': p10s, 'per_query_recall': recalls, 'per_query_ap': aps,
        }

Overwriting evaluator.py


In [20]:
%%writefile queries.py

QUERIES = [
    # ── KEYWORD QUERIES ──────────────────────────────────────────────────────
    {'id':'Q01','type':'keyword','query':'neural network deep learning',
     'challenge':'Short keywords match many docs across AI and Data Science.',
     'relevant_topics':['Artificial Intelligence','Data Science']},
    {'id':'Q02','type':'keyword','query':'blockchain consensus protocol',
     'challenge':'"Consensus" also appears in networking docs.',
     'relevant_topics':['Blockchain','Computer Networks']},
    {'id':'Q03','type':'keyword','query':'quantum entanglement qubit',
     'challenge':'Very specialized; few docs contain all terms together.',
     'relevant_topics':['Quantum Computing']},
    {'id':'Q04','type':'keyword','query':'cloud storage encryption',
     'challenge':'"Encryption" spans Cybersecurity and Cloud Computing.',
     'relevant_topics':['Cloud Computing','Cybersecurity']},
    {'id':'Q05','type':'keyword','query':'IoT sensor network',
     'challenge':'Abbreviation must be expanded; "network" overlaps with Computer Networks.',
     'relevant_topics':['Internet of Things','Computer Networks']},

    # ── NATURAL LANGUAGE QUERIES ──────────────────────────────────────────────
    {'id':'Q06','type':'natural_language',
     'query':'How does machine learning improve cybersecurity threat detection?',
     'challenge':'Cross-topic; long question reduces keyword overlap.',
     'relevant_topics':['Artificial Intelligence','Cybersecurity']},
    {'id':'Q07','type':'natural_language',
     'query':'What are the main challenges of implementing blockchain in supply chains?',
     'challenge':'"Supply chain" may not appear in corpus; needs semantic understanding.',
     'relevant_topics':['Blockchain']},
    {'id':'Q08','type':'natural_language',
     'query':'How do quantum computers outperform classical computers?',
     'challenge':'"Outperform" is not a typical technical keyword.',
     'relevant_topics':['Quantum Computing']},
    {'id':'Q09','type':'natural_language',
     'query':'What is the difference between public and private cloud deployment?',
     'challenge':'Definitional question; docs may not directly compare the two.',
     'relevant_topics':['Cloud Computing']},
    {'id':'Q10','type':'natural_language',
     'query':'How can data science be used to detect network intrusions?',
     'challenge':'Cross-topic bridge between Data Science and Cybersecurity.',
     'relevant_topics':['Data Science','Cybersecurity','Computer Networks']},

    # ── AMBIGUOUS QUERIES ─────────────────────────────────────────────────────
    {'id':'Q11','type':'ambiguous','query':'network security',
     'challenge':'"Network" could mean blockchain, computer network, or neural network.',
     'relevant_topics':['Cybersecurity','Computer Networks','Blockchain']},
    {'id':'Q12','type':'ambiguous','query':'distributed systems',
     'challenge':'Equally relevant to Blockchain, Cloud, and Computer Networks.',
     'relevant_topics':['Blockchain','Cloud Computing','Computer Networks']},
    {'id':'Q13','type':'ambiguous','query':'data mining',
     'challenge':'"Mining" = crypto mining OR data mining technique.',
     'relevant_topics':['Data Science','Blockchain']},
    {'id':'Q14','type':'ambiguous','query':'smart contracts automation',
     'challenge':'"Smart" might pull AI docs; "automation" overlaps with IoT.',
     'relevant_topics':['Blockchain','Internet of Things']},
    {'id':'Q15','type':'ambiguous','query':'edge computing latency',
     'challenge':'"Edge" is key in both IoT and Cloud; "latency" is generic.',
     'relevant_topics':['Internet of Things','Cloud Computing','Computer Networks']},

    # ── NOISY / TYPO QUERIES ──────────────────────────────────────────────────
    {'id':'Q16','type':'noisy','query':'mchine lerning algorythms',
     'challenge':'Three typos — machine, learning, algorithms — all key terms corrupted.',
     'relevant_topics':['Artificial Intelligence','Data Science']},
    {'id':'Q17','type':'noisy','query':'blokchain crytocurency trnsaction',
     'challenge':'Three misspellings in domain-specific vocabulary.',
     'relevant_topics':['Blockchain']},
    {'id':'Q18','type':'noisy','query':'cybr securtiy atack prevntion',
     'challenge':'Heavy noise; all four words contain errors.',
     'relevant_topics':['Cybersecurity']},
    {'id':'Q19','type':'noisy','query':'quantm computng qubit procesing',
     'challenge':'Rare technical vocabulary made harder by misspellings.',
     'relevant_topics':['Quantum Computing']},
    {'id':'Q20','type':'noisy','query':'clod computng servr virtualiztion',
     'challenge':'"clod" instead of "cloud" may confuse spell checker.',
     'relevant_topics':['Cloud Computing']},
]

Overwriting queries.py


In [21]:
%%writefile app.py

import streamlit as st, time
from corpus_loader    import load_corpus
from query_handler    import QueryHandler
from models.vsm       import VSMRetriever
from models.bm25      import BM25Retriever
from models.embedding import EmbeddingRetriever

st.set_page_config(page_title="IR System", page_icon="🔍", layout="wide")

@st.cache_resource(show_spinner=False)
def load_and_index():
    docs  = load_corpus('corpus/corpus.csv')
    vsm   = VSMRetriever();       vsm.fit(docs)
    bm25  = BM25Retriever();      bm25.fit(docs)
    embed = EmbeddingRetriever(); embed.fit(docs)
    return docs, vsm, bm25, embed

def get_snippet(content, query, max_chars=200):
    idx = content.lower().find(query.lower().split()[0])
    if idx == -1: return content[:max_chars] + ' ...'
    start   = max(0, idx - 40)
    snippet = content[start: start + max_chars]
    return ('... ' if start > 0 else '') + snippet + ' ...'

def main():
    st.title("🔍 Information Retrieval System")
    st.caption("500 documents · 8 topics · 3 retrieval models")

    with st.spinner("⏳ Loading corpus and indexing (first run may take ~2-4 min for embeddings)..."):
        docs, vsm, bm25, embed = load_and_index()
    st.success(f"✅ Ready — {len(docs)} documents indexed")

    qh = QueryHandler()

    with st.sidebar:
        st.header("⚙️ Settings")
        model_name = st.selectbox("Retrieval Model",
            ["Vector Space Model (VSM)", "BM25", "Embedding-based"])
        top_k = st.slider("Results to show", 5, 20, 10)
        st.divider()
        st.header("📊 Corpus Topics")
        st.markdown("""
| Topic | Docs |
|---|---|
| Artificial Intelligence | 93 |
| Cloud Computing | 81 |
| Cybersecurity | 67 |
| Blockchain | 60 |
| Data Science | 59 |
| Quantum Computing | 59 |
| Internet of Things | 47 |
| Computer Networks | 46 |
        """)

    query = st.text_input("🔎 Search Query",
        placeholder='Try:  AI security   |   "deep learning"   |   mchine lerning')
    st.caption('Wrap phrases in quotes for exact matching, e.g. `"neural network"`')

    if not query.strip():
        return

    processed = qh.process_query(query)

    with st.expander("🔧 Query Processing Details", expanded=True):
        col1, col2, col3 = st.columns(3)
        col1.markdown(f"**Original:** `{processed['original']}`")
        col2.markdown(f"**Processed:** `{processed['processed']}`")
        if processed['corrections']:
            fixes = ', '.join(f"`{k}` → `{v}`" for k,v in processed['corrections'].items())
            col3.markdown(f"**Spelling fixed:** {fixes}")
        if processed['phrase']:
            st.info(f'📌 Exact phrase mode: **"{processed["phrase"]}"**')

    with st.spinner(f"Searching with {model_name}..."):
        t0 = time.time()
        if   "VSM"   in model_name: results = vsm.search(processed['processed'],   top_k, processed['phrase'])
        elif "BM25"  in model_name: results = bm25.search(processed['processed'],  top_k, processed['phrase'])
        else:                       results = embed.search(processed['processed'],  top_k, processed['phrase'])
        elapsed = time.time() - t0

    st.markdown(f"### 📋 Top {len(results)} Results &nbsp; <small>({elapsed:.3f}s)</small>", unsafe_allow_html=True)

    if not results:
        st.warning("No results found. Try different keywords.")
        return

    for res in results:
        doc = res['doc']
        with st.container():
            left, right = st.columns([6, 1])
            with left:
                st.markdown(f"**#{res['rank']}. [{doc['title']}]({doc['link']})**")
                st.caption(f"📁 {doc['topic']}  ·  🔗 {doc['link']}")
                st.markdown(f"<small style='color:gray'>{get_snippet(doc['text'], processed['processed'])}</small>",
                            unsafe_allow_html=True)
            with right:
                st.metric("Score", f"{res['score']:.4f}")
            st.divider()

if __name__ == "__main__":
    main()

Overwriting app.py


In [22]:
%%writefile run_evaluation.py

import pandas as pd
from corpus_loader    import load_corpus
from query_handler    import QueryHandler
from models.vsm       import VSMRetriever
from models.bm25      import BM25Retriever
from models.embedding import EmbeddingRetriever
from evaluator        import Evaluator
from queries          import QUERIES

def get_relevant_ids(documents, relevant_topics):
    return [d['id'] for d in documents if d['topic'] in relevant_topics]

docs  = load_corpus('corpus/corpus.csv')
vsm   = VSMRetriever();       vsm.fit(docs)
bm25  = BM25Retriever();      bm25.fit(docs)
embed = EmbeddingRetriever(); embed.fit(docs)

qh        = QueryHandler()
evaluator = Evaluator()
models    = {'VSM': vsm, 'BM25': bm25, 'Embedding': embed}
rows      = []

for model_name, model in models.items():
    print(f"\n── Evaluating {model_name} ──")
    qr_list = []
    for q in QUERIES:
        proc    = qh.process_query(q['query'])
        results = model.search(proc['processed'], top_k=10, phrase=proc['phrase'])
        qr_list.append({
            'retrieved_ids': [r['doc']['id'] for r in results],
            'relevant_ids':  get_relevant_ids(docs, q['relevant_topics']),
        })
    res = evaluator.evaluate_model(model_name, qr_list)
    print(f"  Precision@10 : {res['avg_precision@10']:.4f}")
    print(f"  Recall       : {res['avg_recall']:.4f}")
    print(f"  MAP          : {res['MAP']:.4f}")
    rows.append({'Model': model_name,
                 'Precision@10': round(res['avg_precision@10'],4),
                 'Recall':       round(res['avg_recall'],4),
                 'MAP':          round(res['MAP'],4)})

df = pd.DataFrame(rows)
print("\n════ FINAL COMPARISON ════")
print(df.to_string(index=False))
df.to_csv('evaluation_results.csv', index=False)
print("\n✅ Saved → evaluation_results.csv")

Overwriting run_evaluation.py


In [23]:
# Kill anything using port 8501
!fuser -k 8501/tcp

# Optional: verify nothing is running
!lsof -i:8501

from pyngrok import ngrok
import time

# Your auth token
NGROK_TOKEN = "3CUlNMBBgsgvDeRLnGtMaed7LbU_7FCP9rrVgFAUHuM85s5TY"
ngrok.set_auth_token(NGROK_TOKEN)

# Kill ALL previous tunnels/sessions
ngrok.kill()

# Wait a bit so ngrok fully closes
time.sleep(3)

# Start Streamlit
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true &'
)

# Wait for streamlit to fully start
time.sleep(5)

# Create NEW tunnel
public_url = ngrok.connect(addr=8501)

print("🌐 Open your app here →", public_url)

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://cornfield-repeated-tattered.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [24]:
!pip install -q pyngrok streamlit

from pyngrok import ngrok
import time

# Set token
ngrok.set_auth_token("3DRFA1JzdeFszTPMNYXTtCWRC6m_4hPZaCBtQnmTAJD8ZQYFF")

# Start streamlit
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 &'
)

# Wait for streamlit
time.sleep(8)

# Disconnect any old tunnels
ngrok.kill()

# Create COMPLETELY NEW RANDOM tunnel
tunnel = ngrok.connect(
    addr=8501,
    proto="http"
)

print("PUBLIC URL:", tunnel.public_url)

PUBLIC URL: https://creative-walmart-lucid.ngrok-free.dev


In [25]:
# Run this in a SEPARATE cell while the GUI is NOT running
# (or open a new Colab session)
exec(open('run_evaluation.py').read())

Loaded 506 documents.
VSM: Indexed 506 documents.
BM25: Indexed 506 documents.
Embeddings: Loading from cache...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings: Ready. (506 docs)

── Evaluating VSM ──
  Precision@10 : 0.7800
  Recall       : 0.0812
  MAP          : 0.0694

── Evaluating BM25 ──
  Precision@10 : 0.8300
  Recall       : 0.0895
  MAP          : 0.0826

── Evaluating Embedding ──
  Precision@10 : 0.8650
  Recall       : 0.0935
  MAP          : 0.0892

════ FINAL COMPARISON ════
    Model  Precision@10  Recall    MAP
      VSM         0.780  0.0812 0.0694
     BM25         0.830  0.0895 0.0826
Embedding         0.865  0.0935 0.0892

✅ Saved → evaluation_results.csv
